In [1]:
# --- STEP 0: System Setup ---
! git clone https://github.com/aqwertyuiop48/edu-101-java-code.git
print("🔧 Installing Java 17, Temporal CLI, and Maven...")
!sudo apt-get update -y > /dev/null
!sudo apt-get install openjdk-17-jdk-headless maven -y > /dev/null
!ls

Cloning into 'edu-101-java-code'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 148 (delta 38), reused 145 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 30.05 MiB | 14.92 MiB/s, done.
Resolving deltas: 100% (38/38), done.
🔧 Installing Java 17, Temporal CLI, and Maven...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 33.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg

In [2]:
# 1. Download using the correct 2026 URL
!curl -sSf https://temporal.download/cli.sh | sh

# 2. Add the binary to the Python environment's PATH
import os
temporal_bin_path = os.path.expanduser("~/.temporalio/bin")
os.environ['PATH'] += f":{temporal_bin_path}"

# 3. Create a system-wide link so all '!temporal' commands work seamlessly
!ln -sf /root/.temporalio/bin/temporal /usr/local/bin/temporal

# 4. Verify it works!
print("\n✅ Verification:")
!temporal --version

%cd /content/edu-101-java-code/example1

temporal: Downloading Temporal CLI latest
temporal: Temporal CLI installed at /root/.temporalio/bin/temporal
temporal: For convenience, we recommend adding it to your PATH
temporal: If using bash, run echo export PATH="\$PATH:/root/.temporalio/bin" >> ~/.bashrc

✅ Verification:
temporal version 1.6.1 (Server 1.30.1, UI 2.45.3)
/content/edu-101-java-code/example1


## --- EXAMPLE 1 ---

In [3]:
import subprocess
import time

# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 1...")
# Make sure we are in the right folder!
%cd /content/edu-101-java-code/example1

print("🛰️ Terminal 1: Starting Temporal Server (Port 8081)...")
# Popen starts the server in the background and moves to the next line immediately
import subprocess
import time

# 1. Start the Temporal server in the background (equivalent to trailing '&')
print("Starting Temporal server...")
server_process = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8081','--db-filename','cluster2.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")

# Note: The server_process is still running in the background here.
# You can now run your Gradle command via subprocess, or keep the script running.

print("👷 Terminal 2: Compiling...")
!mvn clean compile > /dev/null

print("👷 Terminal 2: Starting Java Worker...")
# Run the worker in the background too
worker_process = subprocess.Popen(['mvn', 'exec:java', '-Dexec.mainClass=helloworkflow.SayHelloWorker'])

print("⏳ Waiting 15 seconds for server and worker to start...")
time.sleep(15)

print("🎬 Terminal 3: Running Starter...")
# We use ! for the starter because we DO want the notebook to wait for this one to finish
!mvn exec:java -Dexec.mainClass="helloworkflow.Starter"

^C

🏗️ Setting up Example 1...
/content/edu-101-java-code/example1
🛰️ Terminal 1: Starting Temporal Server (Port 8081)...
Starting Temporal server...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
👷 Terminal 2: Compiling...
👷 Terminal 2: Starting Java Worker...
⏳ Waiting 15 seconds for server and worker to start...
🎬 Terminal 3: Running Starter...
[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.example:temporal-demo >----------------------
[INFO] Building temporal-demo 1.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- exec-maven-plugin:3.6.3:java (default-cli) @ temporal-demo ---
[helloworkflow.Starter.main()] INFO io.temporal.serviceclient.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}}
Workflow result: Hello Tem

## --- EXAMPLE 2 ---


In [4]:
import subprocess
import time

# Cleanup everything first
!pkill -9 temporal || true
!pkill -f java || true

print("\n🏗️ Setting up Example 2...")
# Move to root, clone, then move inside
%cd /content
# Only clone if the folder doesn't exist yet
![ -d "samples-java" ] || git clone https://github.com/temporalio/samples-java
%cd samples-java

print("🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...")
# Popen runs the server in the background without blocking the cell
# Note: I removed '--port 7235' so it defaults to 7233. This allows the sample code to connect!
server_process_2 = subprocess.Popen(['temporal', 'server', 'start-dev', '--ui-port', '8082','--db-filename','cluster3.db'])

# 2. Wait 5 seconds for the server to initialize (equivalent to 'sleep 5 &&')
print("Waiting 5 seconds for the server to initialize...")
time.sleep(5)

# 3. Create the search attributes (equivalent to the second command)
print("Registering search attributes...")
create_attrs_cmd = [
    'temporal', 'operator', 'search-attribute', 'create',
    '--namespace', 'default',
    '--type', 'Keyword', '--name', 'CustomKeywordField',
    '--type', 'Int', '--name', 'CustomIntField',
    '--type', 'Double', '--name', 'CustomDoubleField',
    '--type', 'Bool', '--name', 'CustomBoolField',
    '--type', 'Datetime', '--name', 'CustomDatetimeField',
    '--type', 'Text', '--name', 'CustomStringField',
    '--type', 'KeywordList', '--name', 'CustomKeywordListField'
]

# Using subprocess.run will wait for this specific command to finish
try:
    subprocess.run(create_attrs_cmd, check=True)
    print("Search attributes registered successfully!")
except subprocess.CalledProcessError as e:
    print(f"Failed to register search attributes: {e}")
print("⏳ Waiting 10 seconds for server to boot...")
time.sleep(10)

print("👷 Terminal 5: Building and Executing HelloAccumulator...")
# We use ! here because we WANT the cell to wait for Gradle to finish and print the result
!./gradlew build > /dev/null
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

^C

🏗️ Setting up Example 2...
/content
Cloning into 'samples-java'...
remote: Enumerating objects: 7772, done.
remote: Counting objects: 100% (2468/2468), done.
remote: Compressing objects: 100% (779/779), done.
remote: Total 7772 (delta 1936), reused 1751 (delta 1651), pack-reused 5304 (from 1)
Receiving objects: 100% (7772/7772), 3.53 MiB | 3.92 MiB/s, done.
Resolving deltas: 100% (3839/3839), done.
/content/samples-java
🛰️ Terminal 4: Starting 2nd Temporal Server (UI 8082)...
Waiting 5 seconds for the server to initialize...
Registering search attributes...
Search attributes registered successfully!
⏳ Waiting 10 seconds for server to boot...
👷 Terminal 5: Building and Executing HelloAccumulator...
Note: Some input files use or override a deprecated API.
Note: Recompile with -Xlint:deprecation for details.
Note: Some input files use unchecked or unsafe operations.
Note: Recompile with -Xlint:unchecked for details.
/content/samples-java/core/src/test/java/io/temporal/samples/hello/He

In [5]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAccumulator

05:30:45.112 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:30:45.873 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=8667@7056a8644c08} 
05:30:45.897 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAccumulatorTaskQueue", namespace="default", identity=8667@7056a8644c08} 
Worker started for task queue: HelloAccumulatorTaskQueue
05:30:46.491 {HelloAccumulatorWorkflow-yellow } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - received greeting-Greeting [greetingText=Missy Robot, bucket=yellow, greetingKey=key-0] 
05:30:46.492 {HelloAccumulatorWorkflow-blue } [signal sendGreeting] INFO  i.t.s.h.HelloAccumulator$AccumulatorWorkflowImpl - rec

In [6]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivity


05:32:54.495 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:32:55.229 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=9303@7056a8644c08} 
05:32:55.254 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityTaskQueue", namespace="default", identity=9303@7056a8644c08} 
05:32:55.835 {HelloActivityWorkflow 89685f6d-f959-3076-803a-1d2733ebb109} [Activity Executor taskQueue="HelloActivityTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
Hello World!


In [7]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityRetry


05:33:02.221 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:03.482 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=9429@7056a8644c08} 
05:33:03.522 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default", identity=9429@7056a8644c08} 
composeGreeting activity is going to fail
05:33:04.669 {HelloActivityWithRetriesWorkflow d487c5f3-c5ac-3f12-a0de-c36606f10575} [Activity Executor taskQueue="HelloActivityWithRetriesTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=d487c5f3-c5ac-3f12-a0de-c36606f10575, activityType=ComposeGreeting, attempt=1 
j

In [8]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloActivityExclusiveChoice


05:33:16.708 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:17.863 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=9583@7056a8644c08} 
05:33:17.902 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloActivityChoiceTaskQueue", namespace="default", identity=9583@7056a8644c08} 
Order results: Ordered 1 Cherries...Ordered 5 Bananas...Ordered 8 Apples...Ordered 4 Oranges...


In [9]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsync


05:33:25.379 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:26.096 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=9713@7056a8644c08} 
05:33:26.123 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityTaskQueue", namespace="default", identity=9713@7056a8644c08} 
Hello World!
Bye World!


In [10]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloParallelActivity


05:33:31.694 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:32.319 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=9836@7056a8644c08} 
05:33:32.352 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloParallelActivityTaskQueue", namespace="default", identity=9836@7056a8644c08} 
Hello John!
Hello Mary!
Hello Michael!
Hello Jennet!


In [11]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncActivityCompletion


05:33:40.762 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:41.463 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=9963@7056a8644c08} 
05:33:41.506 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncActivityCompletionTaskQueue", namespace="default", identity=9963@7056a8644c08} 
Hello World!


In [12]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloAsyncLambda


05:33:47.119 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:47.837 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=10089@7056a8644c08} 
05:33:47.860 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloAsyncLambdaTaskQueue", namespace="default", identity=10089@7056a8644c08} 
Hello World!
Hello World!


In [13]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDetachedCancellationScope


05:33:54.494 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:33:55.807 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=10214@7056a8644c08} 
05:33:55.842 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDetachedCancellationTaskQueue", namespace="default", identity=10214@7056a8644c08} 
05:33:58.744 {HelloDetachedCancellationWorkflow d30d6b1e-d9f3-3cf7-a903-a2b169d8c7ac} [Activity Executor taskQueue="HelloDetachedCancellationTaskQueue", namespace="default": 1] INFO  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity canceled. ActivityId=d30d6b1e-d9f3-3cf7-a903-a2b169d8c7ac, activityType=SayHello, attempt=1 
Goodbye John!


In [14]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloChild


05:34:04.032 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:34:04.671 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloChildTaskQueue", namespace="default", identity=10353@7056a8644c08} 
Hello World!


In [15]:
!timeout 240s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloCron
# taking too long

05:34:12.116 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:34:13.366 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=10471@7056a8644c08} 
05:34:13.420 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloCronTaskQueue", namespace="default", identity=10471@7056a8644c08} 
Started workflow_id: "HelloCronWorkflow"
run_id: "019cd63d-1654-7c85-8b07-2600b387a786"

From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!
From HelloCronWorkflow: Hello World!


In [16]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloDynamic


05:38:11.984 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:38:12.632 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=11565@7056a8644c08} 
05:38:12.657 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloDynamicTaskQueue", namespace="default", identity=11565@7056a8644c08} 
DynamicACT: Hello John from: DynamicWF


In [17]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloEagerWorkflowStart


05:38:17.762 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:38:18.408 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=11679@7056a8644c08} 
05:38:18.443 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default", identity=11679@7056a8644c08} 
05:38:19.168 {HelloEagerWorkflowStartWorkflow d27d8bc4-9e33-33e8-b424-a24a658bb8dc} [Activity Executor taskQueue="HelloEagerWorkflowStartTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloEagerWorkflowStart$GreetingLocalActivitiesImpl - Composing greeting... 
Hello World!


In [18]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloException


05:38:26.392 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:38:27.040 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=11805@7056a8644c08} 
05:38:27.074 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloExceptionTaskQueue", namespace="default", identity=11805@7056a8644c08} 
05:38:27.982 {a8b08449-f250-3ca7-b517-03545f5382cc 72313fc1-2e97-3b24-b536-360b1c138fb1} [Activity Executor taskQueue="HelloExceptionTaskQueue", namespace="default": 1] WARN  i.t.i.a.ActivityTaskExecutors$ActivityTaskExecutor - Activity failure. ActivityId=72313fc1-2e97-3b24-b536-360b1c138fb1, activityType=ComposeGreeting, attempt=1 
java.io.IOException: Hello World!
	at io.temporal.samples.hello.Hel

In [19]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloLocalActivity


05:38:34.028 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:38:34.698 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloLocalActivity", namespace="default", identity=11936@7056a8644c08} 
05:38:34.731 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloLocalActivity", namespace="default", identity=11936@7056a8644c08} 
Hello World!


In [20]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPeriodic


05:38:43.582 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:38:44.286 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=12064@7056a8644c08} 
05:38:44.319 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPeriodicTaskQueue", namespace="default", identity=12064@7056a8644c08} 
Started a new GreetingWorkflow.
Observing the workflow execution for 20 seconds.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4159 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 4940 milliseconds and then I will greet you again.
From HelloPeriodicWorkflow: Hello World! I will sleep for 5915 milliseconds and then I will greet you agai

In [21]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloPolymorphicActivity


05:39:09.829 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:10.915 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=12269@7056a8644c08} 
05:39:10.957 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloPolymorphicActivityTaskQueue", namespace="default", identity=12269@7056a8644c08} 
Hello World!
Bye World!



In [22]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloQuery


05:39:17.725 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:18.399 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloQueryTaskQueue", namespace="default", identity=12399@7056a8644c08} 
Hello World!
Bye World!


In [23]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSaga


05:39:26.350 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:27.475 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=12519@7056a8644c08} 
05:39:27.499 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSagaTaskQueue", namespace="default", identity=12519@7056a8644c08} 
ActivityOperationImpl.execute() is called with amount 10
ActivityOperationImpl.execute() is called with amount 20
Other compensation logic in main workflow.
ActivityCompensationImpl.compensate() is called with amount -20
ActivityCompensationImpl.compensate() is called with amount -10


In [24]:
!timeout 300s ./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSchedules

05:39:35.825 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:39:36.552 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=12656@7056a8644c08} 
05:39:36.580 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloScheduleTaskQueue", namespace="default", identity=12656@7056a8644c08} 
From HelloScheduleWorkflow-2026-03-10T05:39:36Z: Hello World from HelloSchedule scheduled at 2026-03-10T05:39:36Z!
From HelloScheduleWorkflow-2026-03-10T05:39:40Z: Hello World from HelloSchedule scheduled at 2026-03-10T05:39:40Z!
From HelloScheduleWorkflow-2026-03-10T05:39:45Z: Hello World from HelloSchedule scheduled at 2026-03-10T05:39:45Z!
From HelloScheduleWorkflow-2026-03-10T05:39:50Z: Hello World

In [25]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignal


05:40:34.314 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:40:35.159 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalTaskQueue", namespace="default", identity=12985@7056a8644c08} 
[Hello World!, Hello Universe!]


In [26]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSearchAttributes

05:40:40.154 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:40:40.839 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=13103@7056a8644c08} 
05:40:40.896 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSearchAttributesTaskQueue", namespace="default", identity=13103@7056a8644c08} 
In workflow we get CustomKeywordField is: keys
Hello SearchAttributes!


In [27]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloTypedSearchAttributes


05:40:47.992 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:40:49.296 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=13221@7056a8644c08} 
05:40:49.345 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloTypedSearchAttributesTaskQueue", namespace="default", identity=13221@7056a8644c08} 
Hello TypedSearchAttributes how are you doing?


In [28]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSideEffect



05:40:55.205 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:40:55.940 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=13353@7056a8644c08} 
05:40:55.984 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSideEffectTaskQueue", namespace="default", identity=13353@7056a8644c08} 
Hello World
Hello World


In [29]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloUpdate


05:41:01.735 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:41:02.965 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=13471@7056a8644c08} 
05:41:03.010 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloUpdateTaskQueue", namespace="default", identity=13471@7056a8644c08} 
05:41:04.269 {HelloUpdateWorkflow 787c214b-0889-3aea-a836-8fb2d497bf7f} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$GreetingActivitiesImpl - Composing greeting... 
05:41:04.477 {HelloUpdateWorkflow f5861d4e-7eea-3449-bfb7-f1ae3b54a99e} [Activity Executor taskQueue="HelloUpdateTaskQueue", namespace="default": 1] INFO  i.t.s.h.HelloActivity$Greetin

In [30]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithTimer


05:41:11.404 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:41:12.087 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=13609@7056a8644c08} 
05:41:12.115 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloSignalWithTimerTaskQueue", namespace="default", identity=13609@7056a8644c08} 
Processing value: Value 2
Processing value: Value 5
Processing value: Value 7
Processing value: Value 10
05:41:37.683 {HelloSignalWithTimerWorkflow } [workflow-method-HelloSignalWithTimerWorkflow-91e18ded-83fc-48d0-bbb9-7bca3a68c1d2] INFO  i.t.s.h.HelloSignalWithTimer$SignalWithTimerWorkflowImpl - Timer canceled via exit signal 
Processing value: Value 11


In [31]:
!./gradlew -q --console=plain execute -PmainClass=io.temporal.samples.hello.HelloSignalWithStartAndWorkflowInit

05:41:42.694 { } [main] INFO  i.t.s.WorkflowServiceStubsImpl - Created WorkflowServiceStubs for channel: ManagedChannelOrphanWrapper{delegate=ManagedChannelImpl{logId=1, target=127.0.0.1:7233}} 
05:41:43.379 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Workflow Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=13831@7056a8644c08} 
05:41:43.403 { } [main] INFO  i.t.i.worker.MultiThreadedPoller - start: MultiThreadedPoller{name=Activity Poller taskQueue="HelloWithInitTaskQueue", namespace="default", identity=13831@7056a8644c08} 
Result: Hello Michael Jordan,Hello John Stockton
05:41:44.435 {without-init } [signal addGreeting] WARN  i.t.i.sync.WorkflowExecutionHandler - Workflow execution failure WorkflowId='without-init', RunId=502d43da-cf07-4568-b1e5-0a55008ce086, WorkflowType='MyWorkflowNoInit' 
java.lang.NullPointerException: Cannot invoke "java.util.List.add(Object)" because "this.peopleToGreet" is null
	at io.temporal.sam

In [32]:
# Cleanup
!pkill -9 temporal || true
!pkill -f java || true

^C
